In [20]:
from huggingface_hub import snapshot_download
from pathlib import Path

# ============================================================
# Step A — Download Geneformer V2 104M
# ============================================================

model_dir = Path(r"F:\GeneformerProject\models\Geneformer-V2-104M")

model_dir.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id="ctheodoris/Geneformer",
    allow_patterns=[
        "Geneformer-V2-104M/config.json",
        "Geneformer-V2-104M/model.safetensors",
        "Geneformer-V2-104M/generation_config.json",
        "Geneformer-V2-104M/training_args.bin",
    ],
    local_dir=str(model_dir),
)

print("Download completed.")
print("Model directory:")
print(model_dir)


# Verify the files
model_path = Path(
    r"F:\GeneformerProject\models\Geneformer-V2-104M\Geneformer-V2-104M"
)

print("Model directory:", model_path)
print()

for f in model_path.iterdir():
    print(f.name, "->", f.stat().st_size / (1024**2), "MB")

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 334.35it/s]

Download completed.
Model directory:
F:\GeneformerProject\models\Geneformer-V2-104M
Model directory: F:\GeneformerProject\models\Geneformer-V2-104M\Geneformer-V2-104M

config.json -> 0.0005626678466796875 MB
generation_config.json -> 8.58306884765625e-05 MB
model.safetensors -> 398.2268867492676 MB
training_args.bin -> 0.00524139404296875 MB


In [21]:
import torch
from transformers import AutoModel, AutoConfig

# ============================================================
# Step B — Load Geneformer V2 104M
# ============================================================

model_path = Path(
    r"F:\GeneformerProject\models\Geneformer-V2-104M\Geneformer-V2-104M"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Load configuration
config = AutoConfig.from_pretrained(str(model_path))

print("\nModel configuration:")
print("Model type:", config.model_type)
print("Hidden size:", config.hidden_size)
print("Number of layers:", config.num_hidden_layers)
print("Number of attention heads:", config.num_attention_heads)
print("Maximum position embeddings:", config.max_position_embeddings)
print("Vocabulary size:", config.vocab_size)

# Load pretrained model
model = AutoModel.from_pretrained(
    str(model_path),
    config=config
)

model = model.to(device)
model.eval()

print("\n✓ Geneformer V2 104M loaded successfully.")

Some weights of BertModel were not initialized from the model checkpoint at F:\GeneformerProject\models\Geneformer-V2-104M\Geneformer-V2-104M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Device: cuda

Model configuration:
Model type: bert
Hidden size: 768
Number of layers: 12
Number of attention heads: 12
Maximum position embeddings: 4096
Vocabulary size: 20275

✓ Geneformer V2 104M loaded successfully.


In [22]:
#============================================================
# Path setting and number of Genes
#============================================================
dataset_path = r"F:\GeneformerProject\data\geneformer_tokenized\third_strategy\preprocessed_5_cancers\tcga_geneformer_ig_top4094"
output_dir = Path(
    r"F:\GeneformerProject\data\geneformer_embeddings\third_strategy\preprocessed_5_cancers"
)

output_dir.mkdir(parents=True, exist_ok=True)
N_Gens = str(4096)

print("Source Directory:",dataset_path )
print("output Directory:",output_dir)


Source Directory: F:\GeneformerProject\data\geneformer_tokenized\third_strategy\preprocessed_5_cancers\tcga_geneformer_ig_top4094
output Directory: F:\GeneformerProject\data\geneformer_embeddings\third_strategy\preprocessed_5_cancers


In [23]:
# ============================================================
# Step C — Load one TCGA sequence
# ============================================================

from datasets import load_from_disk

ds = load_from_disk(dataset_path)

sample = ds["train"][0]

input_ids = torch.tensor(
    sample["input_ids"],
    dtype=torch.long
).unsqueeze(0).to(device)

print("Input shape:", input_ids.shape)
print("Cancer type:", sample["Cancer_Type"])
print("Label:", sample["labels"])

print("\nFirst 10 tokens:")
print(input_ids[0, :10].tolist())

print("\nLast 10 tokens:")
print(input_ids[0, -10:].tolist())

Input shape: torch.Size([1, 4096])
Cancer type: PRAD
Label: 4

First 10 tokens:
[2, 8128, 11973, 14988, 10099, 10219, 2899, 1272, 3558, 11801]

Last 10 tokens:
[14945, 13113, 2415, 13506, 19757, 12701, 9475, 13519, 15504, 3]


In [24]:
# ============================================================
# Step D — Generate one Geneformer embedding for first sample for checking
# ============================================================
import numpy as np

# ------------------------------------------------------------
# Use the first TCGA training sample
# ------------------------------------------------------------

sample = ds["train"][0]

input_ids = torch.tensor(
    sample["input_ids"],
    dtype=torch.long
).unsqueeze(0).to(device)

# Attention mask: all gens tokens are real tokens
attention_mask = torch.ones_like(input_ids)

print("Input shape:", input_ids.shape)

# ------------------------------------------------------------
# Generate Geneformer hidden states
# ------------------------------------------------------------

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

# Full contextual representation
last_hidden_state = outputs.last_hidden_state

print("Last hidden state shape:", last_hidden_state.shape)

# ------------------------------------------------------------
# Extract CLS embedding
# ------------------------------------------------------------

cls_embedding = last_hidden_state[:, 0, :]

print("CLS embedding shape:", cls_embedding.shape)

# Convert to NumPy
cls_embedding_np = cls_embedding.cpu().numpy()

print("\nFirst 10 embedding values:")
print(cls_embedding_np[0, :10])

print("\nEmbedding dimension:", cls_embedding_np.shape[1])

Input shape: torch.Size([1, 4096])
Last hidden state shape: torch.Size([1, 4096, 768])
CLS embedding shape: torch.Size([1, 768])

First 10 embedding values:
[ 0.08466163  0.1405063   0.53594494  1.5136722  -1.2009348   0.42127907
 -0.5363687   0.7128947   0.2961269  -1.0490059 ]

Embedding dimension: 768


In [25]:
# ============================================================
# Step E — Generate Geneformer CLS embeddings
# ============================================================

import torch
import numpy as np
from pathlib import Path
from datasets import load_from_disk
from tqdm.auto import tqdm
import time
import gc

# ============================================================
# Settings
# ============================================================


BATCH_SIZE = 2

print("Dataset:", dataset_path)
print("Output:", output_dir)
print("Batch size:", BATCH_SIZE)
print("Device:", device)

# ============================================================
# Load dataset
# ============================================================

ds = load_from_disk(str(dataset_path))

print("\nDataset:")
print(ds)

# ============================================================
# Embedding function
# ============================================================

def generate_cls_embeddings(dataset, split_name, batch_size=2):

    n_samples = len(dataset)

    print("\n" + "=" * 60)
    print(f"Processing: {split_name}")
    print(f"Samples: {n_samples}")
    print("=" * 60)

    all_embeddings = []
    all_labels = []
    all_cancer_types = []
    all_tissue_types = []

    start_time = time.time()

    # --------------------------------------------------------
    # Process in batches
    # --------------------------------------------------------

    for start in tqdm(
        range(0, n_samples, batch_size),
        desc=f"{split_name} embeddings"
    ):

        end = min(start + batch_size, n_samples)

        batch = dataset[start:end]

        # Convert input IDs to tensor
        input_ids = torch.tensor(
            batch["input_ids"],
            dtype=torch.long,
            device=device
        )

        # All tokens are real tokens
        attention_mask = torch.ones_like(input_ids)

        # ----------------------------------------------------
        # Geneformer inference
        # ----------------------------------------------------

        with torch.inference_mode():
          with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
          ):
            outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
         )

            # [batch, Ngen, 768]
            hidden_states = outputs.last_hidden_state

            # CLS = first token
            # [batch, 768]
            cls_embeddings = hidden_states[:, 0, :]

        # Move from GPU to CPU
        cls_embeddings = cls_embeddings.cpu().numpy()

        all_embeddings.append(cls_embeddings)

        # Labels and metadata
        all_labels.extend(batch["labels"])
        all_cancer_types.extend(batch["Cancer_Type"])
        all_tissue_types.extend(batch["tissue_type"])

        # Free GPU memory
        del input_ids
        del attention_mask
        del outputs
        del hidden_states
        del cls_embeddings

        torch.cuda.empty_cache()

    # ========================================================
    # Combine batches
    # ========================================================

    embeddings = np.concatenate(all_embeddings, axis=0)

    labels = np.array(all_labels, dtype=np.int64)

    cancer_types = np.array(
        all_cancer_types,
        dtype=object
    )

    tissue_types = np.array(
        all_tissue_types,
        dtype=object
    )

    elapsed = time.time() - start_time

    print("\nCompleted:", split_name)
    print("Embedding shape:", embeddings.shape)
    print("Labels shape:", labels.shape)
    print("Time:", round(elapsed / 60, 2), "minutes")

    # ========================================================
    # Save
    # ========================================================

    np.save(
        output_dir / f"{split_name}_embeddings.npy",
        embeddings
    )

    np.save(
        output_dir / f"{split_name}_labels.npy",
        labels
    )

    np.save(
        output_dir / f"{split_name}_cancer_types.npy",
        cancer_types
    )

    np.save(
        output_dir / f"{split_name}_tissue_types.npy",
        tissue_types
    )

    print("Saved files:")
    print(
        output_dir / f"{split_name}_embeddings.npy"
    )

    return embeddings, labels


# ============================================================
# Generate TRAIN embeddings
# ============================================================

train_embeddings, train_labels = generate_cls_embeddings(
    ds["train"],
    "train"+ N_Gens,
    BATCH_SIZE
)

# ============================================================
# Generate VALIDATION embeddings
# ============================================================

validation_embeddings, validation_labels = generate_cls_embeddings(
    ds["validation"],
    "validation"+ N_Gens,
    BATCH_SIZE
)

# ============================================================
# Generate TEST embeddings
# ============================================================

test_embeddings, test_labels = generate_cls_embeddings(
    ds["test"],
    "test"+ N_Gens,
    BATCH_SIZE
)

# ============================================================
# Final verification
# ============================================================

print("\n" + "=" * 60)
print("FINAL EMBEDDING RESULTS")
print("=" * 60)

print("Train:")
print("  embeddings:", train_embeddings.shape)
print("  labels:    ", train_labels.shape)

print("\nValidation:")
print("  embeddings:", validation_embeddings.shape)
print("  labels:    ", validation_labels.shape)

print("\nTest:")
print("  embeddings:", test_embeddings.shape)
print("  labels:    ", test_labels.shape)

print("\n✓ All embeddings generated.")

Dataset: F:\GeneformerProject\data\geneformer_tokenized\third_strategy\preprocessed_5_cancers\tcga_geneformer_ig_top4094
Output: F:\GeneformerProject\data\geneformer_embeddings\third_strategy\preprocessed_5_cancers
Batch size: 2
Device: cuda

Dataset:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'Cancer_Type', 'tissue_type'],
        num_rows: 2411
    })
    validation: Dataset({
        features: ['input_ids', 'labels', 'Cancer_Type', 'tissue_type'],
        num_rows: 517
    })
    test: Dataset({
        features: ['input_ids', 'labels', 'Cancer_Type', 'tissue_type'],
        num_rows: 517
    })
})

Processing: train4096
Samples: 2411


train4096 embeddings: 100%|██████████| 1206/1206 [03:04<00:00,  6.52it/s]



Completed: train4096
Embedding shape: (2411, 768)
Labels shape: (2411,)
Time: 3.08 minutes
Saved files:
F:\GeneformerProject\data\geneformer_embeddings\third_strategy\preprocessed_5_cancers\train4096_embeddings.npy

Processing: validation4096
Samples: 517


validation4096 embeddings: 100%|██████████| 259/259 [00:37<00:00,  6.85it/s]



Completed: validation4096
Embedding shape: (517, 768)
Labels shape: (517,)
Time: 0.63 minutes
Saved files:
F:\GeneformerProject\data\geneformer_embeddings\third_strategy\preprocessed_5_cancers\validation4096_embeddings.npy

Processing: test4096
Samples: 517


test4096 embeddings: 100%|██████████| 259/259 [00:40<00:00,  6.43it/s]


Completed: test4096
Embedding shape: (517, 768)
Labels shape: (517,)
Time: 0.67 minutes
Saved files:
F:\GeneformerProject\data\geneformer_embeddings\third_strategy\preprocessed_5_cancers\test4096_embeddings.npy

FINAL EMBEDDING RESULTS
Train:
  embeddings: (2411, 768)
  labels:     (2411,)

Validation:
  embeddings: (517, 768)
  labels:     (517,)

Test:
  embeddings: (517, 768)
  labels:     (517,)

✓ All embeddings generated.
